# 04 — Baselines and First Signals

## Objectif

Avant d'entraîner des modèles de Machine Learning, il est essentiel d'établir des stratégies de référence (*baselines*).

L'objectif de ce notebook est d'évaluer plusieurs règles de prédiction simples, fondées soit sur la distribution des classes, soit sur des hypothèses financières telles que le momentum ou le reversal. Ces baselines constituent le niveau minimal de performance que les futurs modèles devront dépasser.

## Rappel du protocole de validation

Les performances sont évaluées en utilisant exactement le protocole défini dans le notebook précédent :

- validation chronologique ;
- fenêtre d'entraînement croissante (*expanding window*) ;
- 4 folds temporels ;
- 120 dates de validation par fold.

Toutes les baselines seront évaluées selon ce même protocole afin de garantir une comparaison équitable et d'éviter tout biais temporel.

## Imports

In [1]:
import sys 
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [2]:
import numpy as np
import pandas as pd
from src.data_loading import load_X_train, load_y_train
from src.validation import create_expanding_window_folds, check_temporal_folds
from src.evaluation import evaluate_baseline_on_folds
from src.baselines import (baseline_majority_class,
                           baseline_momentum,
                           baseline_reversal,
                           baseline_always_0,
                           baseline_always_1,
                           baseline_momentum_agr,
                           baseline_reversal_agr)

## Création des folds

In [3]:
X_train = load_X_train()
y_train = load_y_train()

In [4]:
y_train.head()

,ROW_ID,target
0,0,0.009210
1,1,0.000405
2,2,0.005967
3,3,-0.000542
4,4,-0.002579


In [5]:
X_train.head()

,ROW_ID,TS,ALLOCATION,RET_20,RET_19,RET_18,RET_17,RET_16,RET_15,RET_14,...,SIGNED_VOLUME_8,SIGNED_VOLUME_7,SIGNED_VOLUME_6,SIGNED_VOLUME_5,SIGNED_VOLUME_4,SIGNED_VOLUME_3,SIGNED_VOLUME_2,SIGNED_VOLUME_1,MEDIAN_DAILY_TURNOVER,GROUP
0,0,DATE_0001,ALLOCATION_01,-0.018192,-0.000306,-0.006881,-0.002393,0.000507,-0.001270,-0.002539,...,0.818730,0.941014,0.714129,-0.323847,0.525097,0.363601,-0.219328,NaN,0.096905,1
1,1,DATE_0001,ALLOCATION_02,-0.006394,-0.001059,0.001565,0.000033,0.002829,0.001725,0.000875,...,-1.390336,-0.651784,-0.896826,-0.636931,-1.074450,-0.748884,-0.718912,NaN,0.009974,4
2,2,DATE_0001,ALLOCATION_03,-0.016587,-0.004517,-0.005306,0.004314,0.006471,-0.005868,-0.005030,...,0.961318,0.452482,1.588321,0.790039,1.394445,0.493521,0.268094,NaN,0.044186,1
3,3,DATE_0001,ALLOCATION_04,-0.005344,0.002790,0.006937,-0.004246,-0.005051,-0.000330,-0.000117,...,-0.483377,-0.565114,-0.631710,-0.663300,-1.615905,-0.959046,-0.478789,NaN,0.001150,2
4,4,DATE_0001,ALLOCATION_05,-0.010506,-0.005491,0.007752,-0.012299,0.002191,0.003282,0.000495,...,0.268005,0.757707,1.524626,1.565541,1.563963,1.063209,0.921333,NaN,NaN,4


In [6]:
df_train = X_train.merge(y_train,left_on="ROW_ID",right_on="ROW_ID")
df_train["class"] = np.where(df_train["target"] > 0,1,0)

In [7]:
dates = sorted(list(set(X_train["TS"])))
folds = create_expanding_window_folds(dates)
check_temporal_folds(folds,dates,validation_size=120)

True

In [8]:
folds

[{'fold': 1,
  'train_start': 'DATE_0001',
  'train_end': 'DATE_2042',
  'valid_start': 'DATE_2043',
  'valid_end': 'DATE_2162'},
 {'fold': 2,
  'train_start': 'DATE_0001',
  'train_end': 'DATE_2162',
  'valid_start': 'DATE_2163',
  'valid_end': 'DATE_2282'},
 {'fold': 3,
  'train_start': 'DATE_0001',
  'train_end': 'DATE_2282',
  'valid_start': 'DATE_2283',
  'valid_end': 'DATE_2402'},
 {'fold': 4,
  'train_start': 'DATE_0001',
  'train_end': 'DATE_2402',
  'valid_start': 'DATE_2403',
  'valid_end': 'DATE_2522'}]

In [9]:
df_folds = pd.DataFrame(folds)

In [10]:
df_folds 

,fold,train_start,train_end,valid_start,valid_end
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522


## Baseline 1 — Classe majoritaire

Cette baseline ignore totalement les variables explicatives.

Elle consiste à prédire systématiquement la classe la plus fréquente observée dans le jeu d'entraînement du fold courant.

Bien qu'extrêmement simple, cette stratégie constitue une référence minimale que tout modèle de Machine Learning devra dépasser.

In [11]:
baseline_majority_class_results = evaluate_baseline_on_folds(df_train,folds,baseline_majority_class)
baseline_majority_class_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.522228,24114,12593
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.504837,24396,12316
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.520554,24764,12891
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.522097,25660,13397


## Baseline 2 — Momentum (RET_1)

Le momentum est une hypothèse classique en finance selon laquelle les mouvements récents des prix ont tendance à se prolonger.

Dans cette première version, nous utilisons uniquement la variable `RET_1`.

La règle est volontairement simple :

- si `RET_1` est positif, nous prédisons une classe positive ;
- sinon, nous prédisons une classe négative.

Cette stratégie ne cherche pas à apprendre automatiquement une relation. Elle permet uniquement de tester si une hypothèse financière simple apporte déjà un pouvoir prédictif.

In [12]:
baseline_momentum_results = evaluate_baseline_on_folds(df_train,folds,baseline_momentum)
baseline_momentum_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.517210,24114,12472
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.534145,24396,13031
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.514780,24764,12748
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.519018,25660,13318


## Baseline 3 — Reversal (RET_1)

Le reversal repose sur une hypothèse opposée au momentum.

Il suppose qu'après un mouvement récent important, le marché peut avoir tendance à corriger ce mouvement.

La règle utilisée est donc l'inverse de la baseline momentum :

**Si RET_1 est strictement négatif, nous prédisons une classe positive ; sinon, nous prédisons une classe négative.**


L'objectif n'est pas d'affirmer que cette hypothèse est vraie, mais de comparer expérimentalement deux visions opposées du comportement des marchés.

In [13]:
baseline_reversal_results = evaluate_baseline_on_folds(df_train,folds,baseline_reversal)
baseline_reversal_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.482790,24114,11642
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.465855,24396,11365
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.485220,24764,12016
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.480982,25660,12342


## Baseline 4 — Always 0

In [14]:
baseline_always_0_results = evaluate_baseline_on_folds(df_train,folds,baseline_always_0)
baseline_always_0_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.477772,24114,11521
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.495163,24396,12080
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.479446,24764,11873
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.477903,25660,12263


## Baseline 5 — Always 1

In [15]:
baseline_always_1_results = evaluate_baseline_on_folds(df_train,folds,baseline_always_1)
baseline_always_1_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.522228,24114,12593
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.504837,24396,12316
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.520554,24764,12891
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.522097,25660,13397


La baseline de classe majoritaire produit les mêmes prédictions que la stratégie Always 1 sur les quatre folds, car la classe positive est majoritaire dans chaque ensemble d’entraînement. Les deux stratégies sont conceptuellement différentes : Always 1 est fixe, tandis que la baseline majoritaire adapte sa classe au train du fold.

## Baselines 6 et 7 — Momentum et Reversal agrégés

Les versions agrégées utilisent la moyenne de RET_1, RET_2 et RET_3. L’objectif est de réduire la sensibilité de la règle à un seul rendement récent et de tester si la direction moyenne des trois dernières périodes fournit un signal plus stable. Cette agrégation peut toutefois diluer un signal très récent.

In [16]:
baseline_momentum_agr_results = evaluate_baseline_on_folds(df_train,folds,baseline_momentum_agr)
baseline_momentum_agr_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.515427,24114,12429
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.516109,24396,12591
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.504563,24764,12495
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.502221,25660,12887


In [17]:
baseline_reversal_agr_results = evaluate_baseline_on_folds(df_train,folds,baseline_reversal_agr)
baseline_reversal_agr_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.484573,24114,11685
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.483891,24396,11805
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.495437,24764,12269
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.497779,25660,12773


## Tableaux de comparaison des baselines

In [18]:
pd.DataFrame({
    "fold" : list(df_folds["fold"]),
    "majority" : list(baseline_majority_class_results["accuracy"]),
    "momentum" : list(baseline_momentum_results["accuracy"]),
    "reversal" : list(baseline_reversal_results["accuracy"]),
    "always_0" : list(baseline_always_0_results["accuracy"]),
    "always_1" : list(baseline_always_1_results["accuracy"]),
    "agr_momentum" : list(baseline_momentum_agr_results["accuracy"]),
    "agr_reversal" : list(baseline_reversal_agr_results["accuracy"]),
})

,fold,majority,momentum,reversal,always_0,always_1,agr_momentum,agr_reversal
0,1,0.522228,0.517210,0.482790,0.477772,0.522228,0.515427,0.484573
1,2,0.504837,0.534145,0.465855,0.495163,0.504837,0.516109,0.483891
2,3,0.520554,0.514780,0.485220,0.479446,0.520554,0.504563,0.495437
3,4,0.522097,0.519018,0.480982,0.477903,0.522097,0.502221,0.497779


In [19]:
results = {"majority" : baseline_majority_class_results,
           "momentum" : baseline_momentum_results,
           "reversal" : baseline_reversal_results,
           "always_0" : baseline_always_0_results,
           "always_1" : baseline_always_1_results,
           "agr_momentum" : baseline_momentum_agr_results,
           "agr_reversal" : baseline_reversal_agr_results}

description = pd.DataFrame({
    "majority" : [results["majority"]["accuracy"].mean(),results["majority"]["accuracy"].std(),results["majority"]["accuracy"].min(),results["majority"]["accuracy"].max(),],
    "momentum" : [results["momentum"]["accuracy"].mean(),results["momentum"]["accuracy"].std(),results["momentum"]["accuracy"].min(),results["momentum"]["accuracy"].max()],
    "reversal" : [results["reversal"]["accuracy"].mean(),results["reversal"]["accuracy"].std(),results["reversal"]["accuracy"].min(),results["reversal"]["accuracy"].max()],
    "always_0" : [results["always_0"]["accuracy"].mean(),results["always_0"]["accuracy"].std(),results["always_0"]["accuracy"].min(),results["always_0"]["accuracy"].max()],
    "always_1" : [results["always_1"]["accuracy"].mean(),results["always_1"]["accuracy"].std(),results["always_1"]["accuracy"].min(),results["always_1"]["accuracy"].max()],
    "agr_momentum" : [results["agr_momentum"]["accuracy"].mean(),results["agr_momentum"]["accuracy"].std(),results["agr_momentum"]["accuracy"].min(),results["agr_momentum"]["accuracy"].max()],
    "agr_reversal" : [results["agr_reversal"]["accuracy"].mean(),results["agr_reversal"]["accuracy"].std(),results["agr_reversal"]["accuracy"].min(),results["agr_reversal"]["accuracy"].max()],
},index=["mean_accuracy","std_accuracy","min_accuracy","max_accuracy"])

description = description.T
description = description.reset_index()
description = description.rename(columns={"index":"strategy"})
description

,strategy,mean_accuracy,std_accuracy,min_accuracy,max_accuracy
0,majority,0.517429,0.008429,0.504837,0.522228
1,momentum,0.521288,0.008745,0.514780,0.534145
2,reversal,0.478712,0.008745,0.465855,0.485220
3,always_0,0.482571,0.008429,0.477772,0.495163
4,always_1,0.517429,0.008429,0.504837,0.522228
5,agr_momentum,0.509580,0.007214,0.502221,0.516109
6,agr_reversal,0.490420,0.007214,0.483891,0.497779


La règle momentum fondée sur RET_1 obtient la meilleure accuracy moyenne, avec environ 52,13 %, contre 51,74 % pour la classe majoritaire. Le gain moyen reste limité à environ 0,39 point et n’apparaît que sur un fold sur quatre. Ce résultat suggère un possible signal de persistance à très court terme, mais il n’est ni suffisamment fort ni suffisamment stable pour être considéré comme robuste.

## Conclusion

Dans ce notebook, plusieurs stratégies de référence ont été construites et évaluées.

Ces baselines constituent désormais le niveau minimal de performance que devront dépasser les futurs modèles de Machine Learning.

Elles serviront de point de comparaison tout au long du reste du projet.